# Stuart-Landau Amplitude Dynamics

Coupled phase-amplitude ODE integration via `StuartLandauEngine`.
Demonstrates bifurcation tracking, amplitude-phase coupling, and
PAC (phase-amplitude coupling) analysis.

**Equation:** $\dot{r}_i = (\mu - r_i^2)\,r_i + \varepsilon \sum_j K^r_{ij}\,r_j\,\cos(\theta_j - \theta_i)$

(Acebrón et al. 2005, Rev. Mod. Phys. 77)

In [ ]:
import numpy as np
from scpn_phase_orchestrator.upde.stuart_landau import StuartLandauEngine
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter
from scpn_phase_orchestrator.upde.pac import modulation_index, pac_matrix

## 1. Basic Stuart-Landau Ensemble

8 oscillators with supercritical bifurcation ($\mu > 0$).
Initial amplitudes near zero; they grow to $r \approx \sqrt{\mu}$.

In [ ]:
N = 8
engine = StuartLandauEngine(n_oscillators=N, dt=0.01, method="rk4")

rng = np.random.default_rng(42)
theta = rng.uniform(0, 2 * np.pi, N)
r = rng.uniform(0.01, 0.1, N)
state = np.concatenate([theta, r])

omega = np.ones(N) * 1.0
mu = np.full(N, 0.5)
knm_phase = np.full((N, N), 0.3)
np.fill_diagonal(knm_phase, 0)
knm_r = knm_phase * 0.5
alpha = np.zeros((N, N))

history_R = []
history_r_mean = []

for step in range(1000):
    state = engine.step(
        state, omega, mu, knm_phase, knm_r,
        zeta=0.0, psi=0.0, alpha=alpha, epsilon=0.1,
    )
    phases = state[:N]
    amplitudes = state[N:]
    R, _ = compute_order_parameter(phases)
    history_R.append(R)
    history_r_mean.append(np.mean(amplitudes))

print(f"Final R = {history_R[-1]:.4f}")
print(f"Final mean amplitude = {history_r_mean[-1]:.4f} (expected ~ sqrt(0.5) = {np.sqrt(0.5):.4f})")

## 2. Amplitude and Phase Convergence

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    ax1.plot(history_R, linewidth=0.8)
    ax1.set_ylabel("R (order parameter)")
    ax1.set_title("Phase synchronisation")
    ax1.axhline(y=1.0, color="gray", linestyle="--", linewidth=0.5)

    ax2.plot(history_r_mean, linewidth=0.8, color="tab:orange")
    ax2.axhline(y=np.sqrt(0.5), color="gray", linestyle="--", linewidth=0.5, label=r"$\sqrt{\mu}$")
    ax2.set_ylabel("Mean amplitude")
    ax2.set_xlabel("Step")
    ax2.set_title("Amplitude convergence")
    ax2.legend()

    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

## 3. Phase-Amplitude Coupling (PAC)

Tort et al. 2010 modulation index between low-frequency phase and
high-frequency amplitude.

In [ ]:
n_layers = 4
T = 500
rng2 = np.random.default_rng(99)
phases_history = np.column_stack([np.linspace(0, (i + 1) * 4 * np.pi, T) for i in range(n_layers)])
amps_history = np.column_stack([0.5 + 0.3 * np.cos(phases_history[:, 0]) for _ in range(n_layers)])

pac_mat = pac_matrix(phases_history, amps_history, n_bins=18)
print(f"PAC matrix shape: {pac_mat.shape}")
print(f"PAC matrix:\n{np.array2string(pac_mat, precision=4)}")

## Summary

- `StuartLandauEngine` integrates coupled phase-amplitude ODEs
- Amplitudes converge to $\sqrt{\mu}$ under supercritical bifurcation
- `modulation_index()` quantifies phase-amplitude coupling strength
- `pac_matrix()` provides cross-layer PAC analysis
- All functions auto-delegate to Rust when `spo_kernel` is available